# Notebook 05 - Optimisation des Hyperparamètres

**Projet** : Classification des Astéroïdes Potentiellement Dangereux (PHAs)
**Objectif** : Optimiser les hyperparamètres du meilleur modèle sélectionné dans le notebook précédent.

In [1]:
import pandas as pd
import numpy as np
import joblib
from pathlib import Path
from IPython.display import display

from sklearn.compose import ColumnTransformer
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from sklearn.neural_network import MLPClassifier

from sklearn.model_selection import RandomizedSearchCV, GridSearchCV, StratifiedKFold
import warnings
warnings.filterwarnings("ignore")

RANDOM_STATE = 42
TARGET = "is_potentially_hazardous"
PROJECT_ROOT = Path("..").resolve() if Path.cwd().name == "notebooks" else Path(".").resolve()
DATA_DIR = PROJECT_ROOT / "data" / "processed"
MODELS_DIR = PROJECT_ROOT / "models"

## 1. Chargement et Configuration

In [2]:
train_df = pd.read_csv(DATA_DIR / "train.csv")
X_train = train_df.drop(columns=[TARGET])
y_train = train_df[TARGET]

preprocessor = joblib.load(MODELS_DIR / "preprocessor.joblib")
results_df = pd.read_csv(MODELS_DIR / "modeling_results.csv")

best_model_name = results_df.iloc[0]["Modèle"]
best_strat_name = results_df.iloc[0]["Stratégie"]

print(f"Modèle à optimiser : {best_model_name}")
print(f"Stratégie retenue : {best_strat_name}")

Modèle à optimiser : XGBoost
Stratégie retenue : Baseline


## 2. Justification de la Grille de Recherche

Selon le modèle vainqueur, nous avons sélectionné la grille suivante :
- **Si XGBoost** :
  - `n_estimators` [100, 300, 500] : Permet au gradient boosting de converger, sans exploser les temps de calcul.
  - `max_depth` [3, 5, 7] : On évite les profondeurs de > 10 qui causent du sur-apprentissage très rapide sur XGBoost.
  - `learning_rate` [0.01, 0.05, 0.1] : Valeurs classiques pour stabiliser l'apprentissage avec un nombre modéré d'arbres.
  - `scale_pos_weight` : Au lieu de SMOTE/RUS, on peut utiliser ce paramètre natif ultra-efficace, mais s'il a été sélectionné avec une stratégie, on l'optimise aussi.
- **Si Logistic Regression** : `C` dans [0.01, 0.1, 1, 10] pour réguler le poids des variables colinéaires résiduelles.
- **Si Decision Tree** : `max_depth` restreint à [5, 7, 10] pour lutter contre le sur-apprentissage endémique de l'arbre seul.
- **Si MLPClassifier** : Couches `(64, 32)` ou `(128, 64)`, alpha (L2) `[0.0001, 0.001]`.

In [3]:
# Construction du pipeline de base selon le gagnant
steps = [("preprocessor", preprocessor)]

if best_strat_name == "SMOTE":
    steps.append(("sampler", SMOTE(random_state=RANDOM_STATE)))
elif best_strat_name == "RandomUnderSampler":
    steps.append(("sampler", RandomUnderSampler(random_state=RANDOM_STATE)))

# Paramétrage de la grille
param_grid = {}

if best_model_name == "XGBoost":
    model = XGBClassifier(eval_metric="logloss", random_state=RANDOM_STATE, n_jobs=-1)
    param_grid = {
        "model__n_estimators": [100, 300, 500],
        "model__max_depth": [3, 5, 7],
        "model__learning_rate": [0.01, 0.05, 0.1],
        "model__subsample": [0.8, 1.0],
        "model__colsample_bytree": [0.8, 1.0]
    }
elif best_model_name == "LogisticRegression":
    model = LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)
    if best_strat_name == "Baseline":
        model.set_params(class_weight="balanced")
    param_grid = {
        "model__C": [0.01, 0.1, 1.0, 10.0],
        "model__solver": ["lbfgs", "liblinear"]
    }
elif best_model_name == "DecisionTree":
    model = DecisionTreeClassifier(random_state=RANDOM_STATE)
    if best_strat_name == "Baseline":
        model.set_params(class_weight="balanced")
    param_grid = {
        "model__max_depth": [3, 5, 7, 10],
        "model__min_samples_leaf": [5, 10, 20]
    }
elif best_model_name == "MLPClassifier":
    model = MLPClassifier(max_iter=500, early_stopping=True, random_state=RANDOM_STATE)
    param_grid = {
        "model__hidden_layer_sizes": [(64, 32), (128, 64)],
        "model__alpha": [0.0001, 0.001, 0.01],
        "model__learning_rate_init": [0.001, 0.01]
    }

steps.append(("model", model))
pipeline = ImbPipeline(steps)

## 3. Exécution de la Recherche

Nous utilisons un `RandomizedSearchCV` (ou `GridSearchCV` si petit) pour optimiser les hyperparamètres sur base de la métrique F1-score.

In [4]:
print("Lancement de l'optimisation des hyperparamètres...")

# Si l'espace de recherche est grand, on privilégie RandomizedSearchCV (ex: XGBoost)
if best_model_name == "XGBoost":
    search = RandomizedSearchCV(pipeline, param_distributions=param_grid, n_iter=15, 
                                scoring="f1", cv=StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE),
                                n_jobs=-1, random_state=RANDOM_STATE, verbose=1)
else:
    search = GridSearchCV(pipeline, param_grid=param_grid, 
                          scoring="f1", cv=StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE),
                          n_jobs=-1, verbose=1)

search.fit(X_train, y_train)

print("\nMeilleurs hyperparamètres trouvés :")
for k, v in search.best_params_.items():
    print(f"  {k.replace('model__', '')} : {v}")

print(f"\nScore F1 moyen (CV) du meilleur modèle : {search.best_score_:.4f}")

Lancement de l'optimisation des hyperparamètres...
Fitting 5 folds for each of 15 candidates, totalling 75 fits

Meilleurs hyperparamètres trouvés :
  subsample : 1.0
  n_estimators : 500
  max_depth : 3
  learning_rate : 0.05
  colsample_bytree : 1.0

Score F1 moyen (CV) du meilleur modèle : 0.9897


## 4. Sauvegarde du modèle optimisé

In [5]:
# Sauvegarder l'estimateur final pour l'évaluation
tuned_model = search.best_estimator_
joblib.dump(tuned_model, MODELS_DIR / "tuned_model.joblib")
print("Modèle optimisé sauvegardé dans 'models/tuned_model.joblib'.")

Modèle optimisé sauvegardé dans 'models/tuned_model.joblib'.
